# Lab 2 - Linear regression </br> M1 Data Science, ML1

# Introduction

In [66]:
# %%
# "IPython magic command" to sutomatically reload the imported packages after changes in a package
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import sklearn

# plt.rc("font", **{"family": "sans-serif", "sans-serif": ["Helvetica"]})
# plt.rc("text", usetex=True)

SMALL_SIZE = 16
MEDIUM_SIZE = 20
BIGGER_SIZE = 24

plt.rc("font", size=SMALL_SIZE)  # default text size
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

# display backend for matplotlib
%matplotlib inline
# %matplotlib widget
# %matplotlib notebook

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<div class="alert alert-danger" style="margin-top: 0px">

**To be completed**

Lab report written by: Rami El Hage, Clément Martin, 2025-2026.

</div>

# Exercise 1: linear regression

Let $N, M$ be two postive integers, and consider a dataset $\mathcal{D} = \{ (y_n \mathbf{x}_n) \}_{1 \leq n \leq N}$, with targets $\mathbf{y} = (y_n)_{1 \leq n \leq N} \in \mathbb{R}^N$, and (standardized) data $\mathbf{X} = [\mathbf{x}_1, \dotsc, \mathbf{x}_N]^T \in \mathbb{R}^{N \times M}$.

This exercise is aimed at performing linear regression with `numpy` and `sklearn` on the `California house` dataset, loaded below.

For the sake of simplicity, the dataset is NOT split into a train and a test set until question 5, where the quality of the model is properly assessed.

In [67]:
# loading the dataset
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

# X = housing.data, y = housing.target
X = housing.data
y = housing.target

print(X.shape, y.shape)

print(housing.feature_names)
print(housing.DESCR)

(20640, 8) (20640,)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in

In [68]:
features_min = np.min(X, axis=0)
features_max = np.max(X, axis=0)
range_features = np.abs(features_max - features_min)

with np.printoptions(precision=2):
    print(
        "Feature ranges: \n names: {} \n dr.  : {} \n min  : {} \n max  : {}".format(
            housing.feature_names, range_features, features_min, features_max
        )
    )

Feature ranges: 
 names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'] 
 dr.  : [1.45e+01 5.10e+01 1.41e+02 3.37e+01 3.57e+04 1.24e+03 9.41e+00 1.00e+01] 
 min  : [   0.5     1.      0.85    0.33    3.      0.69   32.54 -124.35] 
 max  : [ 1.50e+01  5.20e+01  1.42e+02  3.41e+01  3.57e+04  1.24e+03  4.20e+01
 -1.14e+02]


<div class="alert" style="margin-top: 0px">

1. Standardize the data using `sklearn`, and store the output into a variable `scaled_X`. Is it useful for this dataset? Justify your answer.

> Indication: take a look at [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) 

</div>

In [69]:
scaler = sklearn.preprocessing.StandardScaler()
scaled_X = scaler.fit_transform(X,y)

<div class="alert alert-warning" style="margin-top: 0px">
Answer question 1:

This method is useful here since the range of the features is very large. The method of standardization improves stability learning process. Transform the dataset into new one with mean=0 and std=1.

</div>

<div class="alert" style="margin-top: 0px">

2. Write ERM problem corresponding to linear least-squares regression. Recall the form of the analytic expression solution, with the definition of the elements involved.

    > Indication: type your answer in $\LaTeX$ in the cell below, or include a picture of your handwritten answer.

</div>

<div class="alert alert-warning" style="margin-top: 0px">
Answer question 2:

$\text{The general ERM formula:} \, \mathcal{R}(\mathcal{D},h) = \sum_{n=1}^N l(Y_{n},h(X_{n}))$

$l(Y_{n},h(X_{n}))=\frac{1}{2}(Y_{n}-h(X_{n}))^{2} \, \text{is the loss function.}$


$\text{D is the dataset, N is the number of examples.
The hypothesis space (set of linear predictors) is:} \, \mathcal{H} = \{ h: \mathbb{R}^{D} \rightarrow \mathbb{R} \,|\, h(x)=\langle \textbf{w},\text{x} \rangle + b, (\textbf{w},b) \in \mathbb{R}^{D} \times \mathbb{R} \}$

\begin{equation*}
    \phi : \mathbb{R}^{D} \to \mathbb{R}^{D+1} 
\end{equation*}
\begin{equation*}
x \mapsto [1,x^{T}]^{T}
\end{equation*}

Reformulation: $\mathcal{H} = \{ h: \mathbb{R}^{D} \rightarrow \mathbb{R} \,|\, h(x)=\langle \phi(\text{x}),\beta \rangle,\beta \in \mathbb{R}^{D+1}\}$

$\text{The solution of this problem is:}\,\beta^{*} \in \underset{\beta\in \mathbb{R}^{D+1}}{\text{argmin}}\frac{1}{2N}||\textbf{y}-\tilde{\textbf{X}}\beta||_{2}^{2}, \, \text{with} \, \tilde{\textbf{X}}= \begin{bmatrix}
    \phi(x_{1})^{T} \\ \vdots \\ \phi(x_{N})^{T}
\end{bmatrix} \in \mathbb{R}^{N\times (D+1)}$

</div>

<div class="alert" style="margin-top: 0px">

3. 
   1. Compute the solution to the problem with `numpy` from the analytic expression, calling the result `beta_numpy`.
   
   2. Compute the solution with `sklearn`, `beta_sklearn`. 

   3. Test whether the two array are significantly different. 

> Hints: 
> - the function [`numpy.linalg.solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html#numpy.linalg.solve) and the class [`sklearn.linear_model.LinearRegression`](https://scikit-learn.org/stable/modules/linear_model.html) will be useful
> - the function [`numpy.allclose`](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html#numpy-allclose) can be useful to test equality between two arrays.

</div>

In [70]:
def phi(x):
    # we create an array consisting of one number which is one and we concatenate it to x
    res = np.array([[1]])
    res = np.concatenate((res, x),axis=1)
    return res

def tild(X):
    # we apply phi(x) on the first row of X then we store it
    a = X[0:1,:]
    X_tild = phi(a)

    # we apply phi(x) on every other rows of X then we concatenate them
    for i in range(1,len(X)): #we need to change 0 to 1
        b = phi(X[i:i+1,:])
        X_tild = np.concatenate((X_tild,b))

    return X_tild

In [71]:
#using numpy

X_tild = tild(scaled_X)
beta_numpy = np.linalg.solve(X_tild.T @ X_tild, X_tild.T @ y)
print(beta_numpy)

#using sklearn
model = sklearn.linear_model.LinearRegression().fit(X_tild, y)
beta_sklearn = model.coef_
print(beta_sklearn)

[ 2.06855817  0.8296193   0.11875165 -0.26552688  0.30569623 -0.004503
 -0.03932627 -0.89988565 -0.870541  ]
[ 0.          0.8296193   0.11875165 -0.26552688  0.30569623 -0.004503
 -0.03932627 -0.89988565 -0.870541  ]


In [72]:
np.allclose(beta_numpy, beta_sklearn)

False

<div class="alert" style="margin-top: 0px">

4. Evaluate $\kappa(\widetilde{\mathbf{X}}^T\widetilde{\mathbf{X}})$, the condition number of the matrix $\widetilde{\mathbf{X}}^T\widetilde{\mathbf{X}}$ using the function [`numpy.linalg.cond`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.cond.html#numpy.linalg.cond). Is the least-squares problem considered well-conditioned? Justify your answer.

</div>

In [73]:
cond = np.linalg.cond(X_tild.T @ X_tild)
cond

np.float64(44.46514975131604)

<div class="alert alert-warning" style="margin-top: 0px">
Answer question 4:

... It is not well-conditionned because 44 >> 1 so the least-squares problem is considered is ill-conditionned

</div>

<div class="alert" style="margin-top: 0px">

5. To properly assess the quality of the regressor, the dataset will now be split into a train and a test set. Complete the code below to:
    - split the dataset into a train and a test set
    - use `sklearn` to apply K-fold cross validation on the train set, and return the regressor associated with the best MSE criterion;
    - evaluate the performance on the test set in terms of the MSE criterion.

    > Indications: 
    > - if needed, take a look at the documentation of [`sklearn.model_selection.train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html#train-test-split), and take some inspiration from [`sklearn` examples](https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols.html#sphx-glr-auto-examples-linear-model-plot-ols-py).
    >
    > - the documentation and examples for [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) (or [`sklearn.pipeline.make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)) and [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) can be useful;
    >
    > - to return the regression coefficients from the pipeline, take a look its [named_steps](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline.named_steps) method (if needed, see also the [stackoverflow answer](https://stackoverflow.com/questions/43856280/return-coefficients-from-pipeline-object-in-sklearn));
    >
    > - for the evaluation metrics, see [the documentation](https://scikit-learn.org/stable/modules/model_evaluation.html) and the module [sklearn.metrics](https://scikit-learn.org/stable/api/sklearn.metrics.html) 
</div> 

In [74]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# TODO: instructions below to be completed
X_train, X_test, y_train, y_test = train_test_split(scaled_X,y,test_size=0.2,random_state=42)


# TODO: instructions below to be completed
# Create a linear regression model
linear_regression = LinearRegression()

# Create scaler
scaler = StandardScaler() 

# TODO: uncomment once code for linear_regression and scaler variables are filled-in
pipe = Pipeline(steps=[("scaler", scaler), ("linearregression", linear_regression)])

# Set-up cross-validation
kf = KFold(n_splits=5)

fold_id = 0
for train_index, test_index in kf.split(X_train):
    X_subtrain, X_subtest = X_train[train_index], X_train[test_index]
    y_subtrain, y_subtest = y_train[train_index], y_train[test_index]

    # train the model using the scaled train set
    # TODO: add missing instruction
    pipe.fit(X_subtrain,y_subtrain)

    # make predictions using the test set
    # TODO: add missing instruction, using the pipe object
    y_pred = pipe.predict(X_subtest)

    # TODO: compute the errors (MSE, MAE, R2), and display these below
    MSE = mean_squared_error(y_subtest,y_pred)
    print("Fold {}: MSE: {:.2f}".format(fold_id, MSE))
    MAE = mean_absolute_error(y_subtest,y_pred)
    print("        MAE: {:.2f}".format(MAE))
    R2 = r2_score(y_subtest,y_pred)
    print("        R2 : {:.2f}".format(R2))
    fold_id += 1

Fold 0: MSE: 0.52
        MAE: 0.53
        R2 : 0.62
Fold 1: MSE: 0.50
        MAE: 0.53
        R2 : 0.61
Fold 2: MSE: 0.52
        MAE: 0.52
        R2 : 0.61
Fold 3: MSE: 0.51
        MAE: 0.52
        R2 : 0.61
Fold 4: MSE: 0.55
        MAE: 0.54
        R2 : 0.60


# Exercise 2: least-squares and matrix inversion

Let $N, M$ be two positive integers, $\mathbf{A} \in \mathbb{R}^{N \times M}$ such that $\text{rank}(\mathbf{A}) = M$, and $\mathbf{y} \in \mathbb{R}^N$. 

This exercise is aimed at comparing the accuracy and speed of 2 approaches to compute $\widehat{\boldsymbol{\beta}} = (\mathbf{A}^T\mathbf{A})^{-1}\mathbf{A}^T\mathbf{y}$, solution to the problem 

\begin{equation*}
    \underset{\boldsymbol{\beta} \in \mathbb{R}^M}{\text{minimize}} \; \frac{1}{2} \| \mathbf{y} - \mathbf{A}\boldsymbol{\beta} \|_2^2
\end{equation*}

1. computing $\mathbf{B} = (\mathbf{A}^T\mathbf{A})^{-1}$, and then $\widehat{\boldsymbol{\beta}} = \mathbf{B}\mathbf{A}^T \mathbf{y}$.
2. solving the problem with an iterative solver, e.g., using [`numpy.linalg.solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html#numpy.linalg.solve).

To do so, we will use the data generated below, knowing the ground truth value $\boldsymbol{\beta}^*$ to evaluate numerical accuracy.

In [75]:
rng = np.random.default_rng(1234)

N = 200
M = 100

A = rng.standard_normal(size=(N, M))
beta_true = rng.standard_normal(size=(M,))
y = A @ beta_true

<div class="alert" style="margin-top: 0px">

1. 
   - Compute $\widehat{\boldsymbol{\beta}}$ with the two approaches mentioned above, registering the time needed to solve the problem with the 2 version. To do so, complete the code cell below.
   - Compare the time required between the two approaches

> Indication: see the functions [`np.linalg.inv`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.inv.html#numpy.linalg.inv) and [`numpy.linalg.solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html#numpy.linalg.solve)

</div>

In [76]:
%%timeit
# Approach based on direct matrix inversion

# TODO: complete instructions below 
B = np.linalg.inv(A.T @ A) 
beta_inv = B @ A.T @ y

295 μs ± 12.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [77]:
%%timeit
# Approach based on iterative algorithm

# TODO: complete instructions below
beta_solve = np.linalg.solve(A.T @ A, A.T @ y)

167 μs ± 6.91 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


<div class="alert" style="margin-top: 0px">

2. The accuracy of $\widehat{\boldsymbol{\beta}}$ can be assess with respect to the ground truth $\boldsymbol{\beta}^*$ in terms of the reconstruction signal to noise ratio (rSNR), expressed in dB, defined by

\begin{equation*}
    \text{rSNR}(\widehat{\boldsymbol{\beta}}, \boldsymbol{\beta}^*) = 10 \log_{10} \Big( \frac{\| \boldsymbol{\beta}^* \|_2^2}{\| \boldsymbol{\beta}^* - \widehat{\boldsymbol{\beta}} \|_2^2} \Big).
\end{equation*}

- Compute the rSNR for the solution `beta_inv` and `beta_solve` computed above, and indicate which of the two esimates is the most precise.
- Conclude on the best approach to adopt to solve linear systems of equations.

</div>

In [78]:
import math

def rSNR(beta_solution, beta_true):
    beta_true_norm = np.linalg.norm(beta_true)
    distance = np.linalg.norm(beta_true - beta_solution)
    return 10 * math.log10((beta_true_norm / distance)**2)

print(rSNR(beta_inv, beta_true))
print(rSNR(beta_solve, beta_true))

295.57496906322285
293.8970666562139


<div class="alert alert-warning" style="margin-top: 0px">
Answer question 2:

The best approach seems that we should solve the problem with an iterative solver since the accuracy is better.

</div>

<div class="alert" style="margin-top: 0px">

3. Use `numpy` to compute the condition number $\kappa(\mathbf{A})$ of the matrix $\mathbf{A}$. What information does this value convey about the inversion problem considered?

> Indication: take a look at the documentation of the function [`numpy.linalg.cond`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.cond.html#numpy.linalg.cond).

</div>

In [79]:
cond = np.linalg.cond(A)
cond

np.float64(5.338572426355569)

<div class="alert alert-warning" style="margin-top: 0px">
Answer question 3:

... This value means that the matrix A is well-conditionned.

</div>

---

End of lab2.